# Caculate F1 score

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchmetrics.classification import F1Score, BinaryF1Score
from sklearn.metrics import mean_absolute_error, accuracy_score
import numpy as np
import os
import pickle
import sys
sys.path.append("../../../")
from ctf_datasets.adni.dataset_SD import bin_array,ordinal_array



# === Main Evaluation Function ===
def evaluate_by_batch(loaded_results, batch_size=None, num_slices=10):
    attributes = list(loaded_results['predictions'].keys())
    results_summary = {}

    for attr in attributes:
        preds = torch.tensor(loaded_results['predictions'][attr]).squeeze()
        targets = torch.tensor(loaded_results['targets'][attr]).squeeze()

        preds = torch.sigmoid(preds)

        if attr == "apoE":
            preds_label = preds.argmax(dim=-1)
            targets_label = bin_array(targets, reverse=True).long()
            metric = F1Score(task="multiclass", num_classes=3).to(preds.device)
        elif attr == "sex":
            preds_label = (preds >= 0.5).int()
            targets_label = targets.int()
            metric = BinaryF1Score().to(preds.device)
        elif attr == "slice":
            preds_label = preds.argmax(dim=-1)
            targets_label = ordinal_array(targets, m=num_slices, reverse=True).long()
            metric = F1Score(task="multiclass", num_classes=num_slices).to(preds.device)
        elif attr in ['age', 'brain_vol', 'vent_vol']:
            metric = nn.L1Loss()
        else:
            continue  # Skip unknown attr

        if attr in ['apoE', 'sex', 'slice']:
            # Classification
            if batch_size is None:
                f1 = metric(preds_label, targets_label).item()
                acc = accuracy_score(targets_label.cpu().numpy(), preds_label.cpu().numpy())
                results_summary[attr] = {
                    'F1-score': round(f1, 3),
                    'Accuracy': round(acc, 3)
                }
            else:
                f1_list, acc_list = [], []
                for i in range(0, len(preds_label), batch_size):
                    p = preds_label[i:i+batch_size]
                    t = targets_label[i:i+batch_size]
                    #if len(torch.unique(t)) > 1:
                    f1 = metric(p, t).item()
                    # else:
                    #     f1 = 1.0 if torch.equal(p, t) else 0.0
                    acc = accuracy_score(t.cpu().numpy(), p.cpu().numpy())
                    f1_list.append(f1)
                    acc_list.append(acc)
                results_summary[attr] = {
                    'F1-score': round(np.mean(f1_list), 3),
                    'F1-std': round(np.std(f1_list), 2),
                    'Accuracy': round(np.mean(acc_list), 3),
                    'Acc-std': round(np.std(acc_list), 2)
                }

        else:
            # Regression
            preds_val = preds.cpu().numpy()
            targets_val = targets.cpu().numpy()
            if batch_size is None:
                l1 = mean_absolute_error(targets_val, preds_val)
                results_summary[attr] = {'L1-loss': round(l1, 3)}
            else:
                l1_list = []
                for i in range(0, len(preds_val), batch_size):
                    p = preds_val[i:i+batch_size]
                    t = targets_val[i:i+batch_size]
                    l1 = mean_absolute_error(t, p)
                    l1_list.append(l1)
                results_summary[attr] = {
                    'L1-loss': round(np.mean(l1_list), 3),
                    'L1-std': round(np.std(l1_list), 2)
                }

    return results_summary


In [ ]:
# Example usage
# attr_keys = ['apoE', 'age', 'sex', 'brain_vol', 'vent_vol', 'slice']

for target_attr in attr

target_attr = 'slice'
result_path = "../saved_benchmark/ADNI/DSCM_effectiveness_step100.0_scale1.0_NTIFalse/{}/final_results.pkl".format(target_attr)
with open(result_path, "rb") as f:
    loaded_results = pickle.load(f)

summary = evaluate_by_batch(loaded_results, batch_size=256, num_slices=10)
for key in summary.keys():
    print("{}:{}".format(key,summary[key]))


apoE:{'F1-score': 0.53, 'F1-std': 0.17, 'Accuracy': 0.53, 'Acc-std': 0.17}
age:{'L1-loss': 0.158, 'L1-std': 0.04}
sex:{'F1-score': 0.626, 'F1-std': 0.18, 'Accuracy': 0.648, 'Acc-std': 0.12}
brain_vol:{'L1-loss': 0.11, 'L1-std': 0.03}
vent_vol:{'L1-loss': 0.033, 'L1-std': 0.01}
slice:{'F1-score': 0.485, 'F1-std': 0.06, 'Accuracy': 0.485, 'Acc-std': 0.06}


In [20]:
# === Evaluation configuration ===
target_attr = "apoE"  # Change as needed
batch_size = 256

# === Root directory of results ===
root_dir = "../saved/ADNI/effectiveness"
attr_keys = ['apoE', 'age', 'sex', 'brain_vol', 'vent_vol', 'slice']

# === Find all subdirectories matching do(attr) ===
do_folders = [name for name in attr_keys if os.path.isdir(os.path.join(root_dir, name))]

print(f"Comparing metrics on attribute '{target_attr}' under different do(.) conditions with batch size = {batch_size}:\n")

for do_attr in do_folders:
    result_path = os.path.join(root_dir, do_attr, "final_results.pkl")
    if not os.path.exists(result_path):
        continue  # Skip if file missing

    with open(result_path, "rb") as f:
        loaded_results = pickle.load(f)

    results = evaluate_by_batch(loaded_results, batch_size=batch_size)

    if target_attr not in results:
        continue  # Skip if attribute not found

    scores = results[target_attr]
    if 'F1-score' in scores:
        print(f"do({do_attr})\t{target_attr}: F1 = {scores['F1-score']} ± {scores.get('F1-std', 0)}, "
              f"Acc = {scores['Accuracy']} ± {scores.get('Acc-std', 0)}")
    else:
        print(f"do({do_attr})\t{target_attr}: L1-loss = {scores['L1-loss']} ± {scores.get('L1-std', 0)}")

Comparing metrics on attribute 'apoE' under different do(.) conditions with batch size = 256:

do(apoE)	apoE: F1 = 0.0 ± 0.0, Acc = 0.0 ± 0.0
do(age)	apoE: F1 = 0.0 ± 0.0, Acc = 0.0 ± 0.0
do(sex)	apoE: F1 = 0.0 ± 0.0, Acc = 0.0 ± 0.0
do(brain_vol)	apoE: F1 = 0.0 ± 0.0, Acc = 0.0 ± 0.0
do(vent_vol)	apoE: F1 = 0.0 ± 0.0, Acc = 0.0 ± 0.0
do(slice)	apoE: F1 = 0.0 ± 0.0, Acc = 0.0 ± 0.0


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchmetrics.classification import F1Score, BinaryF1Score
from sklearn.metrics import mean_absolute_error, accuracy_score
import numpy as np
import os
import pickle
import sys
sys.path.append("../../../")
from ctf_datasets.adni.dataset_SD import bin_array,ordinal_array



# === Main Evaluation Function ===
def evaluate_by_batch(loaded_results, batch_size=None, num_slices=10):
    attributes = list(loaded_results['predictions'].keys())
    results_summary = {}

    for attr in attributes:
        preds = torch.tensor(loaded_results['predictions'][attr]).squeeze()
        targets = torch.tensor(loaded_results['targets'][attr]).squeeze()

        preds = torch.sigmoid(preds)

        if attr == "apoE":
            preds_label = preds.argmax(dim=-1)
            targets_label = bin_array(targets, reverse=True).long()
            metric = F1Score(task="multiclass", num_classes=3).to(preds.device)
        elif attr == "sex":
            preds_label = (preds >= 0.5).int()
            targets_label = targets.int()
            metric = BinaryF1Score().to(preds.device)
        elif attr == "slice":
            preds_label = preds.argmax(dim=-1)
            targets_label = ordinal_array(targets, m=num_slices, reverse=True).long()
            metric = F1Score(task="multiclass", num_classes=num_slices).to(preds.device)
        elif attr in ['age', 'brain_vol', 'vent_vol']:
            metric = nn.L1Loss()
        else:
            continue  # Skip unknown attr

        if attr in ['apoE', 'sex', 'slice']:
            # Classification
            if batch_size is None:
                f1 = metric(preds_label, targets_label).item()
                acc = accuracy_score(targets_label.cpu().numpy(), preds_label.cpu().numpy())
                results_summary[attr] = {
                    'F1-score': round(f1, 3),
                    'Accuracy': round(acc, 3)
                }
            else:
                f1_list, acc_list = [], []
                for i in range(0, len(preds_label), batch_size):
                    p = preds_label[i:i+batch_size]
                    t = targets_label[i:i+batch_size]
                    #if len(torch.unique(t)) > 1:
                    f1 = metric(p, t).item()
                    # else:
                    #     f1 = 1.0 if torch.equal(p, t) else 0.0
                    acc = accuracy_score(t.cpu().numpy(), p.cpu().numpy())
                    f1_list.append(f1)
                    acc_list.append(acc)
                results_summary[attr] = {
                    'F1-score': round(np.mean(f1_list), 3),
                    'F1-std': round(np.std(f1_list), 2),
                    'Accuracy': round(np.mean(acc_list), 3),
                    'Acc-std': round(np.std(acc_list), 2)
                }

        else:
            # Regression
            preds_val = preds.cpu().numpy()
            targets_val = targets.cpu().numpy()
            if batch_size is None:
                l1 = mean_absolute_error(targets_val, preds_val)
                results_summary[attr] = {'L1-loss': round(l1, 3)}
            else:
                l1_list = []
                for i in range(0, len(preds_val), batch_size):
                    p = preds_val[i:i+batch_size]
                    t = targets_val[i:i+batch_size]
                    l1 = mean_absolute_error(t, p)
                    l1_list.append(l1)
                results_summary[attr] = {
                    'L1-loss': round(np.mean(l1_list), 3),
                    'L1-std': round(np.std(l1_list), 2)
                }

    return results_summary



# === 设置评估目标属性 ===
batch_size = 256

# === 根目录 ===
root_dir = "../saved_benchmark/ADNI/200000_DSCM_effectiveness_step100.0_scale1.0_NTIFalse_2025-10-07T16-43-06-controlnet_textcond_nocontrastgeneration_text_global_after"
attr_keys = ['apoE', 'age', 'sex', 'brain_vol', 'vent_vol', 'slice']
# === 遍历所有 do(attr) 文件夹 ===
do_folders = [name for name in attr_keys if os.path.isdir(os.path.join(root_dir, name))]


for target_attr in attr_keys:
    print(f"Comparing '{batch_size}' F1-score on attribute '{target_attr}' under different do(.) conditions:")
    for do_attr in do_folders:
        result_path = os.path.join(root_dir, do_attr, "final_results.pkl")
        if not os.path.exists(result_path):
            continue  # skip missing

        with open(result_path, "rb") as f:
            loaded_results = pickle.load(f)

        results = evaluate_by_batch(loaded_results, batch_size=batch_size)

        if target_attr not in results:
            continue  # skip if target attr not in result

        scores = results[target_attr]

        if target_attr in ['apoE','sex', 'slice']:
            if batch_size is None:
                print(f"do({do_attr}): F1 = {scores['F1-score']}, Acc = {scores['Accuracy']}")
            else:
                print(f"do({do_attr}): F1 = {scores['F1-score']} ± {scores['F1-std']}, Acc = {scores['Accuracy']} ± {scores['Acc-std']}")
        else:
            if batch_size is None:
                print(f"do({do_attr}): MAE = {scores['L1-loss']:.3f}")
            else:
                print(f"do({do_attr}): MAE = {scores['L1-loss']:.3f} ± {scores['L1-std']:.2f}")


Comparing '256' F1-score on attribute 'apoE' under different do(.) conditions:
do(apoE): F1 = 0.218 ± 0.1, Acc = 0.218 ± 0.1
do(age): F1 = 0.549 ± 0.19, Acc = 0.549 ± 0.19
do(sex): F1 = 0.561 ± 0.19, Acc = 0.561 ± 0.19
do(brain_vol): F1 = 0.561 ± 0.19, Acc = 0.561 ± 0.19
do(vent_vol): F1 = 0.561 ± 0.19, Acc = 0.561 ± 0.19
do(slice): F1 = 0.561 ± 0.19, Acc = 0.561 ± 0.19
Comparing '256' F1-score on attribute 'age' under different do(.) conditions:
do(apoE): MAE = 0.147 ± 0.04
do(age): MAE = 0.215 ± 0.01
do(sex): MAE = 0.152 ± 0.05
do(brain_vol): MAE = 0.171 ± 0.04
do(vent_vol): MAE = 0.175 ± 0.04
do(slice): MAE = 0.147 ± 0.04
Comparing '256' F1-score on attribute 'sex' under different do(.) conditions:
do(apoE): F1 = 0.63 ± 0.26, Acc = 0.688 ± 0.13
do(age): F1 = 0.692 ± 0.24, Acc = 0.722 ± 0.15
do(sex): F1 = 0.687 ± 0.24, Acc = 0.714 ± 0.14
do(brain_vol): F1 = 0.435 ± 0.08, Acc = 0.445 ± 0.07
do(vent_vol): F1 = 0.646 ± 0.25, Acc = 0.7 ± 0.14
do(slice): F1 = 0.646 ± 0.25, Acc = 0.7 ± 0.1

In [11]:

for do_attr in do_folders:
    result_path = os.path.join(root_dir, do_attr, "final_results.pkl")
    if not os.path.exists(result_path):
        continue  # skip missing

    with open(result_path, "rb") as f:
        loaded_results = pickle.load(f)
    print('do {} lipips distance is {}'.format(do_attr,np.mean(loaded_results['lpips'])))

do apoE lipips distance is 0.04007621482014656
do age lipips distance is 0.07249633222818375
do sex lipips distance is 0.08214952796697617
do brain_vol lipips distance is 0.08836637437343597
do vent_vol lipips distance is 0.1200912669301033
do slice lipips distance is 0.10331352800130844


In [4]:
scores.keys()

dict_keys(['F1-score', 'F1-std', 'Accuracy', 'Acc-std'])

In [ ]:
#dict_keys(['predictions', 'targets'])
loaded_results.keys()
#dict_keys(['Young', 'Male', 'No_Beard', 'Bald'])
loaded_results['predictions'].keys()

dict_keys(['predictions', 'targets'])

# FID and Minimality

In [1]:
import torch
minimality = torch.load("./saved/celeA_complex/FID/minimality.pt")
countefactual_images =  torch.load("./saved/celeA_complex/FID/counterfactual_tensors.pt")

/tmp/ipykernel_290807/4134258590.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  minimality = torch.load("./saved/celeA_complex/FID/minimality.pt")
/tmp/ipykernel_290807

In [6]:
minimality['factuals'][1].shape

AttributeError: 'tuple' object has no attribute 'shape'